In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ---------- CONFIG ----------
ROWS = 8500
DATE_START = datetime(2022, 1, 1)
DATE_END = datetime(2025, 12, 31)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------- LOAD SOURCE DATA (PASTE YOUR TWO FILES AS CSV FIRST) ----------
# Convert your two txt files to CSV manually or by replacing tabs with commas.
# Save them locally as:
# - Merged_Scout_Activities_source.csv
# - Détails-des-Coûts-de-Camp-Dinars_translated_source.csv

merged_source = pd.read_csv("Merged_Scout_Activities_source.xlsx")
costs_source = pd.read_csv("Détails-des-Coûts-de-Camp-Dinars_translated_source.xlsx")

# ---------- DISTRIBUTIONS ----------
country_weights = merged_source["country"].value_counts(normalize=True).to_dict()
activity_type_weights = merged_source["activity_type"].value_counts(normalize=True).to_dict()
sdg_weights = merged_source["sdg"].value_counts(normalize=True).to_dict()
unit_weights = merged_source["unit"].value_counts(normalize=True).to_dict()

countries = list(country_weights.keys())
country_probs = list(country_weights.values())

activity_types = list(activity_type_weights.keys())
activity_probs = list(activity_type_weights.values())

sdgs = list(sdg_weights.keys())
sdg_probs = list(sdg_weights.values())

units = list(unit_weights.keys())
unit_probs = list(unit_weights.values())

# ---------- HELPERS ----------
def random_date():
    delta = DATE_END - DATE_START
    return DATE_START + timedelta(days=random.randint(0, delta.days))

def random_duration(activity_type):
    if "Camp" in activity_type or "camp" in activity_type:
        return random.randint(3, 14)
    if "Event" in activity_type or "Workshop" in activity_type:
        return random.randint(1, 4)
    if "Meeting" in activity_type or "Assembly" in activity_type:
        return random.randint(1, 2)
    return random.randint(1, 5)

def random_participants(activity_type):
    if "Camp" in activity_type or "camp" in activity_type:
        return random.randint(40, 160)
    if "International" in activity_type:
        return random.randint(80, 200)
    return random.randint(20, 90)

def random_budget(activity_type):
    if "International" in activity_type:
        return round(random.uniform(300000, 4500000), 2)
    if "Camp" in activity_type or "camp" in activity_type:
        return round(random.uniform(1500, 14000), 2)
    return round(random.uniform(100, 6000), 2)

def camp_name_generator():
    prefixes = ["Spring", "Summer", "Winter", "Autumn", "National", "Youth", "Advanced", "Regional"]
    suffixes = ["Camp", "Scout Camp", "Guides Camp", "Rovers Camp", "Adventure Camp", "Outdoor Camp"]
    year = random.randint(2022, 2025)
    return f"{random.choice(prefixes)} {random.choice(suffixes)} {year}"

def cost_breakdown(total):
    # Logical split with random variance
    transport = total * random.uniform(0.15, 0.35)
    food = total * random.uniform(0.25, 0.40)
    equipment = total * random.uniform(0.10, 0.25)
    health = total * random.uniform(0.05, 0.15)
    org = total - (transport + food + equipment + health)
    return [round(transport), round(food), round(equipment), round(health), round(org)]

# ---------- GENERATE CAMP NAMES + TOTALS TO MATCH ----------
camp_rows_needed = int(ROWS * 0.5)
camp_names = [camp_name_generator() for _ in range(camp_rows_needed)]
camp_totals = [random.randint(1800, 14000) for _ in range(camp_rows_needed)]

camp_df = pd.DataFrame({
    "Camp": camp_names,
    "Transportation": [cost_breakdown(t)[0] for t in camp_totals],
    "Food": [cost_breakdown(t)[1] for t in camp_totals],
    "Equipment": [cost_breakdown(t)[2] for t in camp_totals],
    "Health/Safety": [cost_breakdown(t)[3] for t in camp_totals],
    "Organization": [cost_breakdown(t)[4] for t in camp_totals],
    "Total": camp_totals,
    "participant": [random.randint(20, 160) for _ in range(camp_rows_needed)]
})

# ---------- MERGED SCOUT ACTIVITIES ----------
activity_rows = []
for i in range(ROWS):
    if i < camp_rows_needed:
        activity_name = camp_names[i]
        budget = camp_totals[i]
        activity_type = "Camp"
    else:
        activity_name = random.choice(merged_source["activity_name"].dropna().unique())
        activity_type = np.random.choice(activity_types, p=activity_probs)
        budget = random_budget(activity_type)

    row = {
        "activity_name": activity_name,
        "activity_type": activity_type,
        "country": np.random.choice(countries, p=country_probs),
        "date": random_date().strftime("%d/%m/%Y"),
        "duration": random_duration(activity_type),
        "participants": random_participants(activity_type),
        "budget": budget,
        "sdg": np.random.choice(sdgs, p=sdg_probs),
        "unit": np.random.choice(units, p=unit_probs),
    }
    activity_rows.append(row)

merged_generated = pd.DataFrame(activity_rows)

# ---------- COSTS FILE ----------
# Ensure total rows = 8500 by adding more random camp rows if needed
extra_needed = ROWS - len(camp_df)
if extra_needed > 0:
    extra_names = [camp_name_generator() for _ in range(extra_needed)]
    extra_totals = [random.randint(1800, 14000) for _ in range(extra_needed)]
    extra_df = pd.DataFrame({
        "Camp": extra_names,
        "Transportation": [cost_breakdown(t)[0] for t in extra_totals],
        "Food": [cost_breakdown(t)[1] for t in extra_totals],
        "Equipment": [cost_breakdown(t)[2] for t in extra_totals],
        "Health/Safety": [cost_breakdown(t)[3] for t in extra_totals],
        "Organization": [cost_breakdown(t)[4] for t in extra_totals],
        "Total": extra_totals,
        "participant": [random.randint(20, 160) for _ in range(extra_needed)]
    })
    camp_df = pd.concat([camp_df, extra_df], ignore_index=True)

# ---------- EXPORT ----------
merged_generated.to_excel("Merged_Scout_Activities_generated.xlsx", index=False)
camp_df.to_excel("Détails-des-Coûts-de-Camp-Dinars_translated_generated.xlsx", index=False)

print("✅ Files created:")
print(" - Merged_Scout_Activities_generated.xlsx")
print(" - Détails-des-Coûts-de-Camp-Dinars_translated_generated.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'Merged_Scout_Activities_source.xlsx'